In [2]:
# Import config
from pathlib import Path
import sys

p = Path.cwd()
for _ in range(6):
    if (p / "config.py").exists():
        sys.path.insert(0, str(p))
        break
    p = p.parent

from config import *

# create main dirs + language subfolders
ensure_dirs(
    DATA_DIR, RAW_DIR, RESULT_DIR, MODELS_DIR, LOG_DIR, CHARACTER_DIR, TEST_DATA_DIR
)
ensure_dirs(*(RAW_DIR / lang for lang in LANGUAGES))
ensure_dirs(*(RESULT_DIR / lang for lang in LANGUAGES))

# Define data/result directory

In [ ]:
import os

data_folder = TEST_DATA_DIR
result_folder = RESULT_DIR
manga_list = "vi/Almark/Vol. 1 Ch. 1"  # Will be changed to list later

# Almark
manga_folder = os.path.join(data_folder, manga_list)

individual_result_folder = os.path.join(result_folder, manga_list)
json_output_dir = os.path.join(individual_result_folder, "json_results")
result_image_output_dir = os.path.join(individual_result_folder, MAGI_IMAGE_RESULT)

cut_bubbles_dir = os.path.join(individual_result_folder, CUT_BUBBLES)
os.makedirs(cut_bubbles_dir, exist_ok=True)  # Create the directory if it doesn't exist

raw_images = os.listdir(manga_folder)
json_files = os.listdir(json_output_dir)

# Cut essential text boxes using JSON coor

In [ ]:
from PIL import Image
import json

for json_file in json_files[:]:
    if not json_file.endswith(".json"):
        continue  # skip non-json files

    base_name = os.path.splitext(json_file)[0]
    image_candidates = [
        f
        for f in raw_images
        if f.startswith(base_name) and f.lower().endswith((".png", ".jpg", ".jpeg"))
    ]

    try:
        image_file = image_candidates[0]  # pick first match
    except IndexError:
        print(f"⚠️ No matching image found for {json_file}, skipping...")
        continue
    else:
        # load image & json
        image_path = os.path.join(manga_folder, image_file)
        json_path = os.path.join(json_output_dir, json_file)

        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        image = Image.open(image_path).convert("RGB")

        # cut essential texts
        for idx, (bbox, is_essential) in enumerate(
            zip(data["texts"], data["is_essential_text"])
        ):
            if not is_essential:
                continue
            x1, y1, x2, y2 = map(int, bbox)
            # Calculate width and height of the box
            w = x2 - x1
            h = y2 - y1
            # Extend by 10% of width and height on all sides
            dw = int(w * 0.1)
            dh = int(h * 0.1)
            x1 = max(0, x1 - dw)
            y1 = max(0, y1 - dh)
            x2 = min(image.width, x2 + dw)
            y2 = min(image.height, y2 + dh)
            cropped = image.crop((x1, y1, x2, y2))

            # save to page-specific folder
            page_dir = os.path.join(cut_bubbles_dir, base_name)
            os.makedirs(page_dir, exist_ok=True)
            out_path = os.path.join(page_dir, f"{base_name}_{idx:03}.png")
            cropped.convert("L").save(out_path)  # Convert to grayscale before saving

        print(f"✅ Cut texts saved for {json_file} in {cut_bubbles_dir}")

✅ Cut texts saved for 00.json in D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\cut_texts
✅ Cut texts saved for 01.json in D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\cut_texts
✅ Cut texts saved for 02.json in D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\cut_texts
✅ Cut texts saved for 03.json in D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\cut_texts
✅ Cut texts saved for 04.json in D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\cut_texts
✅ Cut texts saved for 05.json in D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\cut_texts
✅ Cut texts saved for 06.json in D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\cut_texts
✅ Cut texts saved for 07.json in D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-

# Cut essential + inessential text boxes using JSON coordinates

In [ ]:
from PIL import Image
import json

for json_file in json_files[:]:
    if not json_file.endswith(".json"):
        continue  # skip non-json files

    base_name = os.path.splitext(json_file)[0]
    image_candidates = [
        f
        for f in raw_images
        if f.startswith(base_name) and f.lower().endswith((".png", ".jpg", ".jpeg"))
    ]

    try:
        image_file = image_candidates[0]  # pick first match
    except IndexError:
        print(f"⚠️ No matching image found for {json_file}, skipping...")
        continue
    else:
        # load image & json
        image_path = os.path.join(manga_folder, image_file)
        json_path = os.path.join(json_output_dir, json_file)

        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        image = Image.open(image_path).convert("RGB")

        # cut both essential and inessential texts
        for idx, (bbox, is_essential) in enumerate(
            zip(data["texts"], data["is_essential_text"])
        ):
            x1, y1, x2, y2 = map(int, bbox)
            # Calculate width and height of the box
            w = x2 - x1
            h = y2 - y1
            # Extend by 10% of width and height on all sides
            dw = int(w * 0.1)
            dh = int(h * 0.1)
            x1 = max(0, x1 - dw)
            y1 = max(0, y1 - dh)
            x2 = min(image.width, x2 + dw)
            y2 = min(image.height, y2 + dh)
            cropped = image.crop((x1, y1, x2, y2))

            # save to page-specific folder
            page_dir = os.path.join(cut_bubbles_dir, base_name)
            os.makedirs(page_dir, exist_ok=True)
            out_path = os.path.join(page_dir, f"{base_name}_{idx:03}.png")
            cropped.convert("L").save(out_path)  # Convert to grayscale before saving

        print(f"✅ Cut texts saved for {json_file} in {cut_bubbles_dir}")

✅ Cut texts saved for 00.json in D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\cut_bubbles
✅ Cut texts saved for 01.json in D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\cut_bubbles
✅ Cut texts saved for 02.json in D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\cut_bubbles
✅ Cut texts saved for 03.json in D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\cut_bubbles
✅ Cut texts saved for 04.json in D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\cut_bubbles
✅ Cut texts saved for 05.json in D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\cut_bubbles
✅ Cut texts saved for 06.json in D:\Downloads\Tu_Lieu\Cao_Hoc\Master_Thesis\master-thesis\results\vi/Almark/Vol. 1 Ch. 1\cut_bubbles
✅ Cut texts saved for 07.json in D:\Downloads\Tu_Lieu\Cao_Hoc\Master_